# 4. Explanations and robustness checks

Part 4 of 5. Notebook 3 measured how well the model works. This notebook covers the questions a reviewer is likely to ask:
1. What do the study homes look like? (Table 1)
2. Why does electricity alone not reveal efficiency?
3. Has the model only learned the manufacturer?
4. How does it do on a manufacturer it has never seen?
5. Do half-hourly data help, do homes with missing data change the result, and does the 2.5 threshold matter?

Runtime is about 2 minutes with Ridge.

## Step 1: Setup

The same three models as notebook 3, so whichever was chosen as the main model is used here.

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')

import os
DATA_DIR = '/content/drive/MyDrive/EoH_project'      # the raw EoH files
OUT_DIR = DATA_DIR + '/q1_paper'                     # everything this paper series makes
FIG_DIR = OUT_DIR + '/figures'
TABLE_DIR = OUT_DIR + '/tables'
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(TABLE_DIR, exist_ok=True)

import json
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import rankdata
from sklearn.linear_model import RidgeCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.metrics import r2_score
from lightgbm import LGBMRegressor
warnings.filterwarnings('ignore')

plt.rcParams.update({'font.size': 10, 'axes.titlesize': 11, 'axes.labelsize': 10, 'legend.fontsize': 8,
                     'axes.spines.top': False, 'axes.spines.right': False})
BLUE = '#2f6f9f'
ORANGE = '#c96a14'
GREY = '#8a8f98'
GREEN = '#3f8f5a'
MAKER = {'Daikin': 'Manufacturer A', 'Mitsubishi': 'Manufacturer B', 'Vaillant': 'Manufacturer C'}
MAKER_COLOR = {'Daikin': '#2f6f9f', 'Mitsubishi': '#8a5aa8', 'Vaillant': '#5b8c5a'}

B = 2000
REPEATS = 10
inner_cv = KFold(n_splits=5, shuffle=True, random_state=0)


def make_ridge():
    return make_pipeline(StandardScaler(), RidgeCV(alphas=[0.1, 1, 3, 10, 30, 100, 300]))


def make_forest():
    forest = RandomForestRegressor(n_estimators=200, random_state=0, n_jobs=1)
    settings = {'min_samples_leaf': [5, 10, 20], 'max_features': [0.33, 1.0]}
    return GridSearchCV(forest, settings, cv=inner_cv, scoring='neg_mean_squared_error', n_jobs=-1)


def make_lightgbm():
    booster = LGBMRegressor(n_estimators=300, learning_rate=0.03, subsample=0.8, subsample_freq=1,
                            colsample_bytree=0.8, random_state=0, n_jobs=1, verbose=-1)
    settings = {'num_leaves': [3, 7], 'min_child_samples': [10, 20, 40]}
    return GridSearchCV(booster, settings, cv=inner_cv, scoring='neg_mean_squared_error', n_jobs=-1)


makers = {'Ridge': make_ridge, 'Random forest': make_forest, 'LightGBM': make_lightgbm}
print('ready')

## Step 2: Read results from notebooks 2 and 3

`q1_table.csv, q1_input_sets.json, q1_predictions.csv, q1_main_model.json`

As a check, the predictions are rebuilt here and must match notebook 3 exactly.

In [ ]:
data = pd.read_csv(OUT_DIR + '/q1_table.csv')
with open(OUT_DIR + '/q1_input_sets.json') as f:
    input_sets = json.load(f)
with open(OUT_DIR + '/q1_main_model.json') as f:
    MAIN_MODEL = json.load(f)['main_model']
saved = pd.read_csv(OUT_DIR + '/q1_predictions.csv')
assert (saved['Property_ID'].values == data['Property_ID'].values).all()
saved_pred = saved[[f'repeat_{r}' for r in range(REPEATS)]].values

spf = data['SPF'].values
weak = spf < 2.5
data['Manufacturer'] = data['HP_Brand'].map(MAKER)
print('homes:', len(data), '| main model:', MAIN_MODEL)


def precision_flagged(pred, weak_part, share=0.25):
    n_flag = int(round(share * len(pred)))
    flagged = np.argsort(pred, kind='stable')[:n_flag]
    return 100 * weak_part[flagged].mean()


def scores(pred_matrix, spf_part, idx=None):
    # mean over repeats of (precision of flagged homes, rank correlation)
    if idx is None:
        idx = np.arange(len(spf_part))
    spf_rank = rankdata(spf_part[idx])
    weak_part = spf_part[idx] < 2.5
    precisions = []
    ranks = []
    for r in range(pred_matrix.shape[1]):
        p = pred_matrix[idx, r]
        precisions.append(precision_flagged(p, weak_part))
        ranks.append(np.corrcoef(spf_rank, rankdata(p))[0, 1])
    return np.mean(precisions), np.mean(ranks)


def ci(values):
    return np.percentile(values, 2.5), np.percentile(values, 97.5)


def boot_idx(n, seed):
    return np.random.default_rng(seed).integers(0, n, size=(B, n))


def cv_predictions(d, inputs):
    pred = np.zeros((len(d), REPEATS))
    for r in range(REPEATS):
        for train_idx, test_idx in KFold(n_splits=5, shuffle=True, random_state=r).split(d):
            train = d.iloc[train_idx]
            test = d.iloc[test_idx]
            medians = train[inputs].median()
            model = makers[MAIN_MODEL]()
            model.fit(train[inputs].fillna(medians).fillna(0), train['SPF'])
            pred[test_idx, r] = model.predict(test[inputs].fillna(medians).fillna(0))
    return pred


check = cv_predictions(data, input_sets['both'])
print('largest difference from notebook 3 predictions:', round(np.abs(check - saved_pred).max(), 6))

## Step 3: Table 1, study homes

One row per manufacturer plus a total row. Manufacturers are anonymised as A, B and C. Columns: homes, share high-temperature, median floor area, median capacity, median annual electricity, median SPF and share weak.

In [ ]:
def overview(d):
    return pd.Series({'homes': len(d),
                      'high temperature (%)': 100 * d['is_high_temp'].mean(),
                      'median floor area (m2)': d['Total_Floor_Area'].median(),
                      'median HP size (kW)': d['HP_Size_kW'].median(),
                      'median electricity (kWh/year)': d['annual_kWh'].median(),
                      'median SPF': d['SPF'].median(),
                      'weak (%)': 100 * d['weak'].mean()})


rows = {}
for maker, d in data.groupby('Manufacturer'):
    rows[maker] = overview(d)
rows['All homes'] = overview(data)
table1 = pd.DataFrame(rows).T
table1.round(1)

## Step 4: Why electricity alone does not reveal efficiency

From the official annual figures:

`electricity = heat delivered / SPF`

Taking logs turns the division into a subtraction, so the spread of electricity splits into a heat part and an efficiency part:

`var(log electricity) = var(log heat) + var(log SPF) - 2 cov(log heat, log SPF)`

Heat delivered is used here only to explain the result. It is not an input to any model.

(a) annual electricity against SPF
(b) annual electricity against heat delivered
(c) design annual heat against measured heat (black line: equal). A few homes with very large design values fall outside the axes; their number is printed.

In [ ]:
d = data.copy()
d['heat_kWh'] = (d['HP_Energy_Output_selected_window'] + d['IH_Energy_Consumed_selected_window'].fillna(0)
                 + d['BUH_Energy_Consumed_selected_window'].fillna(0))
d['elec_kWh'] = d['Whole_System_Energy_Consumed_selected_window']
d = d.dropna(subset=['heat_kWh', 'elec_kWh', 'SPF'])
with_design = d.dropna(subset=['design_heat_kWh'])


def spearman(a, b):
    return np.corrcoef(rankdata(a), rankdata(b))[0, 1]


log_e = np.log(d['elec_kWh'])
log_q = np.log(d['heat_kWh'])
log_s = np.log(d['SPF'])
parts = {'annual electricity': log_e.var(),
         'heat delivered': log_q.var(),
         'efficiency (SPF)': log_s.var(),
         'minus 2 x covariance': -2 * np.cov(log_q, log_s)[0, 1]}
print('homes:', len(d), '| check heat / electricity = SPF, largest difference:',
      round((d['heat_kWh'] / d['elec_kWh'] - d['SPF']).abs().max(), 4))
print('spread between homes (variance of the log):')
for k, v in parts.items():
    print(f'   {k:22s}: {v:.3f}')
print('heat spread is', round(parts['heat delivered'] / parts['efficiency (SPF)'], 1), 'times the efficiency spread')

rho_a = spearman(d['SPF'], d['elec_kWh'])
rho_b = spearman(d['heat_kWh'], d['elec_kWh'])
rho_c = spearman(with_design['heat_kWh'], with_design['design_heat_kWh'])
ratio = with_design['design_heat_kWh'] / with_design['heat_kWh']
print('rank corr: electricity vs SPF', round(rho_a, 3), '| electricity vs heat', round(rho_b, 3),
      '| design heat vs real heat', round(rho_c, 3))
print('design heat / real heat: median', round(ratio.median(), 2), '| 10% to 90% of homes:',
      round(ratio.quantile(0.1), 2), 'to', round(ratio.quantile(0.9), 2))
top = max(with_design['heat_kWh'].quantile(0.99), with_design['design_heat_kWh'].quantile(0.99))
print('panel (c): homes outside the plotted area:', ((with_design['heat_kWh'] > top) | (with_design['design_heat_kWh'] > top)).sum())

fig = plt.figure(figsize=(12, 3.9))
plt.subplot(1, 3, 1)
plt.scatter(d['SPF'], d['elec_kWh'], s=8, alpha=0.6, color=BLUE)
plt.xlabel('Measured SPF')
plt.ylabel('Annual electricity (kWh)')
plt.title(f'(a) Electricity vs efficiency\nrank corr = {rho_a:.2f}')
plt.subplot(1, 3, 2)
plt.scatter(d['heat_kWh'], d['elec_kWh'], s=8, alpha=0.6, color=BLUE)
plt.xlabel('Annual heat delivered (kWh)')
plt.ylabel('Annual electricity (kWh)')
plt.title(f'(b) Electricity vs heat delivered\nrank corr = {rho_b:.2f}')
plt.subplot(1, 3, 3)
plt.scatter(with_design['heat_kWh'], with_design['design_heat_kWh'], s=8, alpha=0.6, color=ORANGE)
plt.plot([0, top], [0, top], color='black', linewidth=1)
plt.xlim(0, top)
plt.ylim(0, top)
plt.xlabel('Annual heat delivered, measured (kWh)')
plt.ylabel('Annual heat, design estimate (kWh)')
plt.title(f'(c) Design estimate vs measured heat\nrank corr = {rho_c:.2f}')
plt.tight_layout()
plt.show()

why = pd.Series({**{'variance of log ' + k: v for k, v in parts.items()},
                 'rank corr electricity vs SPF': rho_a, 'rank corr electricity vs heat': rho_b,
                 'rank corr design heat vs real heat': rho_c, 'design heat / real heat, median': ratio.median()})

# ---- save + download this figure (comment out the 2 lines below after the first run) ----
fig.savefig(FIG_DIR + '/q1_04_step04_why_electricity.png', dpi=300, bbox_inches='tight')
files.download(FIG_DIR + '/q1_04_step04_why_electricity.png')

## Step 5: Has the model only learned the manufacturer?

Weak shares differ a lot between manufacturers, and manufacturer is one of the inputs. To separate the two, flagging is repeated within each manufacturer: the 25% of that manufacturer's homes with the lowest predicted SPF are flagged, using the unseen-home predictions from notebook 3.

Columns: homes, weak share (random flagging within the manufacturer), flagged precision, gain over random in percentage points, and within-manufacturer rank correlation, all with 95% intervals. An interval fully above zero means the model knows more than the manufacturer name.

In [ ]:
def within_maker(pred_matrix, label, seed):
    rows = []
    for k, brand in enumerate(['Daikin', 'Mitsubishi', 'Vaillant']):
        one = (data['HP_Brand'] == brand).values
        spf_b = spf[one]
        mat = pred_matrix[one]
        random_rate = 100 * (spf_b < 2.5).mean()
        point = scores(mat, spf_b)
        values = []
        for i in boot_idx(one.sum(), seed + k):
            s = scores(mat, spf_b, i)
            values.append([s[0] - 100 * (spf_b[i] < 2.5).mean(), s[1], s[0]])
        values = np.array(values)
        rows.append({'model': label, 'manufacturer': MAKER[brand], 'homes': int(one.sum()),
                     'random (%)': random_rate, 'flagged weak (%)': point[0],
                     'flagged low': ci(values[:, 2])[0], 'flagged high': ci(values[:, 2])[1],
                     'gain (points)': point[0] - random_rate, 'gain low': ci(values[:, 0])[0], 'gain high': ci(values[:, 0])[1],
                     'rank corr': point[1], 'rank low': ci(values[:, 1])[0], 'rank high': ci(values[:, 1])[1]})
    return pd.DataFrame(rows)


within = within_maker(saved_pred, 'trained on all manufacturers', seed=10)
within.round(2)

## Step 6: A manufacturer the model has never seen

The model is trained on two manufacturers and tested on all homes of the third, three times in turn. Manufacturer inputs are dropped because they mean nothing for a new brand. This mirrors a utility applying the model to heat pumps from a manufacturer that was not in the training data.

In [ ]:
inputs_no_brand = [c for c in input_sets['both'] if c not in ['brand_Daikin', 'brand_Mitsubishi']]
unseen_pred = np.zeros((len(data), 1))
for brand in ['Daikin', 'Mitsubishi', 'Vaillant']:
    test_mask = (data['HP_Brand'] == brand).values
    train = data[~test_mask]
    test = data[test_mask]
    medians = train[inputs_no_brand].median()
    model = makers[MAIN_MODEL]()
    model.fit(train[inputs_no_brand].fillna(medians).fillna(0), train['SPF'])
    unseen_pred[test_mask, 0] = model.predict(test[inputs_no_brand].fillna(medians).fillna(0))
    print(MAKER[brand], 'left out: done')

unseen = within_maker(unseen_pred, 'manufacturer never seen', seed=20)
unseen.round(2)

Figure: for each manufacturer, the weak share (grey), flagged precision when the model has seen all manufacturers (blue) and when it has never seen this one (orange), with 95% intervals.

In [ ]:
makers_order = ['Manufacturer A', 'Manufacturer B', 'Manufacturer C']
x = np.arange(3)
w = within.set_index('manufacturer').loc[makers_order]
u = unseen.set_index('manufacturer').loc[makers_order]

fig = plt.figure(figsize=(8, 4))
plt.bar(x - 0.27, w['random (%)'], width=0.26, color=GREY, label='Random flagging within manufacturer')
plt.bar(x, w['flagged weak (%)'], width=0.26, color=BLUE, label='Model trained on all manufacturers',
        yerr=[w['flagged weak (%)'] - w['flagged low'], w['flagged high'] - w['flagged weak (%)']], capsize=3)
plt.bar(x + 0.27, u['flagged weak (%)'], width=0.26, color=ORANGE, label='Manufacturer never seen in training',
        yerr=[u['flagged weak (%)'] - u['flagged low'], u['flagged high'] - u['flagged weak (%)']], capsize=3)
plt.xticks(x, [f'{m}\n({n} homes)' for m, n in zip(makers_order, w['homes'])])
plt.ylabel('Flagged homes with a weak heat pump (%)')
plt.title('Flagging the lowest 25% within each manufacturer')
plt.legend(loc='upper right')
plt.show()

# ---- save + download this figure (comment out the 2 lines below after the first run) ----
fig.savefig(FIG_DIR + '/q1_04_step06_within_manufacturer.png', dpi=300, bbox_inches='tight')
files.download(FIG_DIR + '/q1_04_step06_within_manufacturer.png')

## Step 7: Do half-hourly data add anything?

An earlier version of this work built seven operating features from half-hourly data that daily data cannot provide:

`running_share_heating, starts_per_day_heating, running_power_kW, cold_power_kW, night_share, running_power_per_HP, cold_power_per_HP`

They are read from `EoH_project/q1_features_428.csv`, which is not produced by these notebooks. The main model is compared with and without them, and the step is skipped if the file is not found.

In [ ]:
hh_inputs = ['running_share_heating', 'starts_per_day_heating', 'running_power_kW', 'cold_power_kW', 'night_share',
             'running_power_per_HP', 'cold_power_per_HP']
old_path = DATA_DIR + '/q1_features_428.csv'
hh_table = None
if os.path.exists(old_path):
    old = pd.read_csv(old_path, usecols=['Property_ID'] + hh_inputs)
    data_hh = data.merge(old, on='Property_ID', how='left')
    print('homes with half-hourly inputs:', data_hh[hh_inputs[0]].notna().sum(), 'of', len(data_hh))
    pred_hh = cv_predictions(data_hh, input_sets['both'] + hh_inputs)
    idx = boot_idx(len(data), seed=30)
    base_boot = np.array([scores(saved_pred, spf, i) for i in idx])
    hh_boot = np.array([scores(pred_hh, spf, i) for i in idx])
    diff = hh_boot - base_boot
    rows = []
    for label, pred in [('daily (main)', saved_pred), ('daily + half-hourly', pred_hh)]:
        p, rc = scores(pred, spf)
        rows.append({'inputs': label, 'precision (%)': p, 'rank corr': rc,
                     'R2 test': np.mean([r2_score(spf, pred[:, r]) for r in range(REPEATS)])})
    point = np.array(scores(pred_hh, spf)) - np.array(scores(saved_pred, spf))
    rows.append({'inputs': 'change', 'precision (%)': point[0], 'rank corr': point[1],
                 'precision low': ci(diff[:, 0])[0], 'precision high': ci(diff[:, 0])[1],
                 'rank low': ci(diff[:, 1])[0], 'rank high': ci(diff[:, 1])[1]})
    hh_table = pd.DataFrame(rows).set_index('inputs')
    display(hh_table.round(3))
else:
    print('file not found, step skipped:', old_path)

## Step 8: Homes with complete data only

The full test (5 folds, 10 repeats) is repeated on homes with at least 90% valid days. Similar results mean missing data did not drive the findings.

In [ ]:
full_data = data[data['coverage_%'] >= 90].reset_index(drop=True)
spf_full = full_data['SPF'].values
pred_full = cv_predictions(full_data, input_sets['both'])
p_full, rc_full = scores(pred_full, spf_full)
p_all, rc_all = scores(saved_pred, spf)
coverage_table = pd.DataFrame({
    'all homes': {'homes': len(data), 'random (%)': 100 * weak.mean(), 'precision (%)': p_all, 'rank corr': rc_all,
                  'R2 test': np.mean([r2_score(spf, saved_pred[:, r]) for r in range(REPEATS)])},
    'coverage >= 90%': {'homes': len(full_data), 'random (%)': 100 * (spf_full < 2.5).mean(), 'precision (%)': p_full,
                        'rank corr': rc_full, 'R2 test': np.mean([r2_score(spf_full, pred_full[:, r]) for r in range(REPEATS)])}}).T
coverage_table.round(3)

## Step 8b: Does the 2.5 threshold matter?

The weak threshold comes from EU and UK rules but is still a convention, so results are recalculated for thresholds of 2.3, 2.4, 2.5, 2.6 and 2.7. The model predicts SPF itself, so its predictions do not change with the threshold; only the weak labels do. Rank correlation and R2 do not depend on it.

For each threshold the table shows the number and share of weak homes, flagged precision for the main model and the manufacturer-only baseline, their difference with a 95% interval, and average precision. Average precision summarises ranking quality without fixing the flagged share; for random flagging it equals the weak share. Ties within a manufacturer are broken at random.

Figure: (a) flagged precision by threshold, main model in blue with a 95% band, manufacturer only in grey, random as a dashed line. (b) Average precision.

In [ ]:
# ---- Step 8b: does the result depend on the 2.5 threshold? ----
from sklearn.metrics import average_precision_score

thresholds = [2.3, 2.4, 2.5, 2.6, 2.7]
n_flag = int(round(0.25 * len(data)))
idx = boot_idx(len(data), seed=40)

# manufacturer-only baseline; all homes of one manufacturer get the same prediction, so ties are broken at random
maker_pred = cv_predictions(data, ['brand_Daikin', 'brand_Mitsubishi'])
maker_pred = maker_pred + np.random.default_rng(0).normal(0, 1e-9, maker_pred.shape)


def precision_at_25(pred_matrix, weak_part, rows=None):
    if rows is None:
        rows = np.arange(len(weak_part))
    return np.mean([precision_flagged(pred_matrix[rows, r], weak_part[rows]) for r in range(pred_matrix.shape[1])])


def avg_precision(pred_matrix, weak_part):
    return np.mean([average_precision_score(weak_part, -pred_matrix[:, r]) for r in range(pred_matrix.shape[1])])


rows = []
for t in thresholds:
    weak_t = spf < t
    model_p = precision_at_25(saved_pred, weak_t)
    maker_p = precision_at_25(maker_pred, weak_t)
    boot_model = np.array([precision_at_25(saved_pred, weak_t, i) for i in idx])
    boot_maker = np.array([precision_at_25(maker_pred, weak_t, i) for i in idx])
    boot_diff = boot_model - boot_maker
    rows.append({'threshold': t, 'weak homes': int(weak_t.sum()), 'random (%)': 100 * weak_t.mean(),
                 'model (%)': model_p, 'low': ci(boot_model)[0], 'high': ci(boot_model)[1],
                 'times random': model_p / (100 * weak_t.mean()),
                 'weak found (%)': model_p * n_flag / weak_t.sum(),
                 'manufacturer only (%)': maker_p,
                 'model minus manufacturer': model_p - maker_p, 'diff low': ci(boot_diff)[0], 'diff high': ci(boot_diff)[1],
                 'AP model': avg_precision(saved_pred, weak_t), 'AP manufacturer': avg_precision(maker_pred, weak_t),
                 'AP random': weak_t.mean()})
    print('threshold', t, ': done')
threshold_table = pd.DataFrame(rows).set_index('threshold')
display(threshold_table.round(3))

x = threshold_table.index.values
fig = plt.figure(figsize=(10, 3.8))
plt.subplot(1, 2, 1)
plt.fill_between(x, threshold_table['low'], threshold_table['high'], color=BLUE, alpha=0.15)
plt.plot(x, threshold_table['model (%)'], marker='o', color=BLUE, label='Main model (95% interval shaded)')
plt.plot(x, threshold_table['manufacturer only (%)'], marker='s', color=GREY, label='Manufacturer only')
plt.plot(x, threshold_table['random (%)'], linestyle='--', color='black', label='Random flagging')
plt.xticks(x)
plt.xlabel('Threshold for a weak heat pump (SPF)')
plt.ylabel('Flagged homes that are weak (%)')
plt.title('(a) Flagging the lowest 25%')
plt.legend(loc='upper left')
plt.subplot(1, 2, 2)
plt.plot(x, threshold_table['AP model'], marker='o', color=BLUE, label='Main model')
plt.plot(x, threshold_table['AP manufacturer'], marker='s', color=GREY, label='Manufacturer only')
plt.plot(x, threshold_table['AP random'], linestyle='--', color='black', label='Random')
plt.xticks(x)
plt.xlabel('Threshold for a weak heat pump (SPF)')
plt.ylabel('Average precision')
plt.title('(b) Ranking quality for weak heat pumps')
plt.legend(loc='upper left')
plt.tight_layout()
plt.show()

threshold_table.to_csv(TABLE_DIR + '/q1_04_threshold_check.csv')
print('saved q1_04_threshold_check.csv')

# ---- save + download this figure (comment out the 2 lines below after the first run) ----
fig.savefig(FIG_DIR + '/q1_04_step08b_threshold_check.png', dpi=300, bbox_inches='tight')
files.download(FIG_DIR + '/q1_04_step08b_threshold_check.png')

## Step 9: Save tables

In [ ]:
table1.to_csv(TABLE_DIR + '/q1_04_table1_homes.csv')
why.to_frame('value').to_csv(TABLE_DIR + '/q1_04_why_electricity.csv')
pd.concat([within, unseen]).to_csv(TABLE_DIR + '/q1_04_within_manufacturer.csv', index=False)
coverage_table.to_csv(TABLE_DIR + '/q1_04_coverage_check.csv')
if hh_table is not None:
    hh_table.to_csv(TABLE_DIR + '/q1_04_halfhourly_check.csv')
print('saved')